In [16]:
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import numpy as np
import scipy.stats as ss
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [4]:
df = pd.read_csv('../Dados_Saeb_2021/DADOS/dados_final.csv')
df.drop(columns= [
    'Unnamed: 0',
    'ID_UF',
    'ID_ALUNO',
    'TX_RESP_BLOCO1_LP',
    'TX_RESP_BLOCO2_LP',
    'TX_RESP_BLOCO1_MT',
    'TX_RESP_BLOCO2_MT',
    'ERRO_PADRAO_LP_SAEB',
    'ERRO_PADRAO_MT_SAEB',
    'NU_TIPO_NIVEL_INSE',
    'PESO_ALUNO_INSE'], inplace=True)
for col in df.columns:
    print(col)

ID_AREA
IN_PUBLICA
ID_LOCALIZACAO
PROFICIENCIA_LP_SAEB
PROFICIENCIA_MT_SAEB
TX_RESP_Q01
TX_RESP_Q02
TX_RESP_Q03
TX_RESP_Q04
TX_RESP_Q05
TX_RESP_Q07
TX_RESP_Q08
TX_RESP_Q09a
TX_RESP_Q09b
TX_RESP_Q09c
TX_RESP_Q09d
TX_RESP_Q09e
TX_RESP_Q09f
TX_RESP_Q11a
TX_RESP_Q11b
TX_RESP_Q11c
TX_RESP_Q11d
TX_RESP_Q11e
TX_RESP_Q11f
TX_RESP_Q11g
TX_RESP_Q11h
TX_RESP_Q12a
TX_RESP_Q12b
TX_RESP_Q12c
TX_RESP_Q12d
TX_RESP_Q12e
TX_RESP_Q12f
TX_RESP_Q12g
TX_RESP_Q12h
TX_RESP_Q12i
TX_RESP_Q13
TX_RESP_Q14
TX_RESP_Q15
TX_RESP_Q16
TX_RESP_Q17
TX_RESP_Q18
TX_RESP_Q19
TX_RESP_Q20a
TX_RESP_Q20b
TX_RESP_Q20c
TX_RESP_Q20d
TX_RESP_Q20e
TX_RESP_Q21
TX_RESP_Q22
TX_RESP_Q23a
TX_RESP_Q23b
TX_RESP_Q23c
TX_RESP_Q23d
TX_RESP_Q23e
TX_RESP_Q23f
TX_RESP_Q23g
TX_RESP_Q23h
TX_RESP_Q23i
TX_RESP_Q06
TX_RESP_Q10
TX_RESP_Q12


In [21]:
df_cramer = df.drop(columns=[
    'PROFICIENCIA_LP_SAEB',
    'PROFICIENCIA_MT_SAEB'
])

# Função para calcular o V de Cramer
def cramers_v(x, y):
    confusion_matrix = pd.crosstab(x, y)
    chi2 = ss.chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n-1))
    rcorr = r - ((r-1)**2)/(n-1)
    kcorr = k - ((k-1)**2)/(n-1)
    denominator = min((kcorr-1), (rcorr-1))
    if denominator == 0:
        return 0
    else:
        return np.sqrt(phi2corr / denominator)

cols = df_cramer.columns
cramers_v_matrix = pd.DataFrame(np.zeros((len(cols), len(cols))), index=cols, columns=cols)
for col1 in cols:
    for col2 in cols:
        cramers_v_matrix.loc[col1, col2] = cramers_v(df_cramer[col1], df_cramer[col2])

threshold = 0
high_corr_series = cramers_v_matrix.stack()
high_corr_pairs = high_corr_series[(high_corr_series > threshold) & (high_corr_series < 1)]

# Transforma em DataFrame
high_corr_df = high_corr_pairs.to_frame('v_cramer').reset_index()
high_corr_df.columns = ['Variavel_1', 'Variavel_2', 'v_cramer']
high_corr_df['sorted_vars'] = high_corr_df.apply(lambda row: tuple(sorted((row['Variavel_1'], row['Variavel_2']))), axis=1)
high_corr_df = high_corr_df.drop_duplicates(subset='sorted_vars').drop(columns='sorted_vars').sort_values(by='v_cramer', ascending=False)
high_corr_df = high_corr_df[high_corr_df['Variavel_1'].str[:11] != high_corr_df['Variavel_2'].str[:11]]

print(f"--- Pares de variáveis com V de Cramer > {threshold} (sem auto-relações) ---")
if high_corr_df.empty:
    print("Nenhum par de variáveis encontrado acima do limite.")
else:
    print(high_corr_df.to_string(index=False))
    high_corr_df.to_excel('../analise_preliminar/cramers_v_resultados.xlsx', index=False)

--- Pares de variáveis com V de Cramer > 0 (sem auto-relações) ---
    Variavel_1     Variavel_2  v_cramer
   TX_RESP_Q14    TX_RESP_Q15  0.716089
   TX_RESP_Q02    TX_RESP_Q18  0.500469
   TX_RESP_Q13    TX_RESP_Q15  0.385232
  TX_RESP_Q11d   TX_RESP_Q12c  0.382558
  TX_RESP_Q11g   TX_RESP_Q12i  0.375803
   TX_RESP_Q13    TX_RESP_Q14  0.371891
  TX_RESP_Q09c   TX_RESP_Q23h  0.360129
  TX_RESP_Q12c    TX_RESP_Q06  0.348855
  TX_RESP_Q12d   TX_RESP_Q23g  0.314044
  TX_RESP_Q11g   TX_RESP_Q12f  0.305094
   TX_RESP_Q01   TX_RESP_Q20c  0.301613
  TX_RESP_Q09d   TX_RESP_Q23h  0.299410
   TX_RESP_Q07    TX_RESP_Q08  0.297364
  TX_RESP_Q11h    TX_RESP_Q06  0.296868
  TX_RESP_Q11d    TX_RESP_Q12  0.296516
  TX_RESP_Q11e   TX_RESP_Q12a  0.287039
       ID_AREA    TX_RESP_Q14  0.275565
  TX_RESP_Q11c   TX_RESP_Q12f  0.273390
  TX_RESP_Q09b   TX_RESP_Q23h  0.268785
  TX_RESP_Q11g    TX_RESP_Q12  0.266956
  TX_RESP_Q12b   TX_RESP_Q23b  0.266698
  TX_RESP_Q09e   TX_RESP_Q23h  0.263678
  TX_RESP_Q11

F → mede a intensidade da diferença entre grupos.

p-valor → indica se a diferença é estatisticamente significativa.

eta² → indica o quanto da nota é explicado pela categórica (medida da força da associação).

In [20]:
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

# variáveis dependentes (contínuas)
dependentes = ["PROFICIENCIA_LP_SAEB", "PROFICIENCIA_MT_SAEB"]

# variáveis independentes (categóricas)
categoricas = [
    "TX_RESP_Q01","TX_RESP_Q02","TX_RESP_Q03","TX_RESP_Q04","TX_RESP_Q05","TX_RESP_Q07",
    "TX_RESP_Q08","TX_RESP_Q09a","TX_RESP_Q09b","TX_RESP_Q09c","TX_RESP_Q09d","TX_RESP_Q09e","TX_RESP_Q09f",
    "TX_RESP_Q11a","TX_RESP_Q11b","TX_RESP_Q11c","TX_RESP_Q11d","TX_RESP_Q11e","TX_RESP_Q11f","TX_RESP_Q11g","TX_RESP_Q11h",
    "TX_RESP_Q12a","TX_RESP_Q12b","TX_RESP_Q12c","TX_RESP_Q12d","TX_RESP_Q12e","TX_RESP_Q12f","TX_RESP_Q12g","TX_RESP_Q12h","TX_RESP_Q12i",
    "TX_RESP_Q13","TX_RESP_Q14","TX_RESP_Q15","TX_RESP_Q16","TX_RESP_Q17","TX_RESP_Q18","TX_RESP_Q19",
    "TX_RESP_Q20a","TX_RESP_Q20b","TX_RESP_Q20c","TX_RESP_Q20d","TX_RESP_Q20e",
    "TX_RESP_Q21","TX_RESP_Q22",
    "TX_RESP_Q23a","TX_RESP_Q23b","TX_RESP_Q23c","TX_RESP_Q23d","TX_RESP_Q23e","TX_RESP_Q23f","TX_RESP_Q23g","TX_RESP_Q23h","TX_RESP_Q23i",
    "TX_RESP_Q06","TX_RESP_Q10","TX_RESP_Q12"
]

resultados = []

# Loop pelas variáveis dependentes e categóricas
for dep in dependentes:
    for cat in categoricas:
        try:
            # modelo ANOVA
            modelo = smf.ols(f"{dep} ~ C({cat})", data=df, missing='drop').fit()
            anova_tabela = sm.stats.anova_lm(modelo, typ=2)

            # extrair F e p-valor
            f_val = anova_tabela["F"][0]
            p_val = anova_tabela["PR(>F)"][0]

            # calcular eta² = SS_between / SS_total
            ss_between = anova_tabela["sum_sq"][0]
            ss_total = anova_tabela["sum_sq"].sum()
            eta_sq = ss_between / ss_total if ss_total > 0 else None

            resultados.append({
                "Dependente": dep,
                "Categórica": cat,
                "F": f_val,
                "p-valor": p_val,
                "eta²": eta_sq
            })
        except Exception as e:
            resultados.append({
                "Dependente": dep,
                "Categórica": cat,
                "F": None,
                "p-valor": None,
                "eta²": None
            })

# transformar em DataFrame para análise
df_resultados = pd.DataFrame(resultados)

# ordenar por eta² para ver quem explica mais variância
df_resultados = df_resultados.sort_values(by="eta²", ascending=False)

# mostrar resultados
df_resultados.to_excel('../analise_preliminar/anova_resultados.xlsx', index=False)
